# 🚚 CASO 1 — LOGÍSTICA
## Visualización de Datos con Streamlit
### Notebook Explicativo Completo — Guiado por el instructor

---

**Empresa:** TransCargo Analytics  
**Contexto:** TransCargo S.A. es una empresa de logística colombiana que opera rutas de carga entre las principales ciudades del país. El equipo directivo necesita un **dashboard interactivo** que permita monitorear el desempeño de sus envíos, identificar retrasos, y optimizar la asignación de transportistas.

**Objetivo:** Construir paso a paso una aplicación Streamlit con visualizaciones interactivas usando Plotly, entendiendo cada línea de código.

---
> 📖 **Patrón de lectura F:** Los conceptos clave están al inicio de cada sección. Lee el encabezado, luego el primer párrafo completo, luego los puntos destacados.

---
## 📦 PASO 1 — Instalación y librerías

Antes de iniciar, verificamos que las librerías necesarias estén instaladas.

| Librería | Para qué sirve |
|---|---|
| `streamlit` | Crear la aplicación web interactiva |
| `pandas` | Manipular y analizar el dataset |
| `plotly` | Crear gráficas interactivas |
| `numpy` | Operaciones numéricas |
| `os` | Manejar rutas de archivos del sistema |

In [ ]:
# ─────────────────────────────────────────────────────────────
# INSTALACIÓN (ejecutar UNA vez en Anaconda Prompt si es necesario)
# pip install streamlit plotly
# ─────────────────────────────────────────────────────────────

import pandas as pd          # Análisis y manipulación de datos
import numpy as np           # Operaciones numéricas
import plotly.express as px  # Gráficas interactivas de alto nivel
import plotly.graph_objects as go  # Gráficas de bajo nivel (más control)
import os                    # Manejo de rutas del sistema de archivos

# Verificar versiones
print('✅ pandas    :', pd.__version__)
print('✅ numpy     :', np.__version__)
print('✅ plotly    : importado correctamente')
print('🎉 ¡Todo listo para comenzar!')

---
## 📊 PASO 2 — Cargar y explorar el dataset

Siempre comenzamos explorando los datos antes de visualizarlos. Esto nos permite entender:
- ¿Cuántos registros y columnas tenemos?
- ¿Qué tipo de datos hay en cada columna?
- ¿Hay valores nulos o inconsistentes?

In [ ]:
# ─────────────────────────────────────────────────────────────
# pd.read_csv() → lee un archivo CSV y lo convierte en DataFrame
# Un DataFrame es una tabla con filas y columnas, similar a Excel
# ─────────────────────────────────────────────────────────────
df = pd.read_csv('caso1_logistica_dataset.csv')

# pd.to_datetime() → convierte texto a tipo fecha para poder operar con él
# Sin esto, la columna sería tratada como texto y no podríamos filtrar por mes
df['fecha_despacho'] = pd.to_datetime(df['fecha_despacho'])

# .shape devuelve (filas, columnas)
print(f'📋 Filas   : {df.shape[0]}')
print(f'📋 Columnas: {df.shape[1]}')
print('\n🔍 Primeras 5 filas:')
df.head()  # .head(n) muestra las primeras n filas (por defecto 5)

In [ ]:
# ─────────────────────────────────────────────────────────────
# .info() → muestra el tipo de dato de cada columna y cuántos
# valores no nulos tiene. Es el primer diagnóstico del dataset.
# ─────────────────────────────────────────────────────────────
print('📊 Información del dataset:')
df.info()

In [ ]:
# ─────────────────────────────────────────────────────────────
# .describe() → estadísticas descriptivas de columnas numéricas:
# count, mean (promedio), std (desviación estándar),
# min, 25%, 50% (mediana), 75%, max
# ─────────────────────────────────────────────────────────────
print('📈 Estadísticas descriptivas:')
df.describe().round(2)

In [ ]:
# ─────────────────────────────────────────────────────────────
# .unique() → valores únicos de una columna categórica
# .value_counts() → conteo de cada valor único
# ─────────────────────────────────────────────────────────────
columnas_cat = ['ciudad_origen', 'transportista', 'tipo_carga', 'estado']

for col in columnas_cat:
    print(f'\n🏷️  {col}:')
    print(df[col].value_counts().to_string())

---
## 📈 PASO 3 — Análisis exploratorio con Pandas

Calculamos los indicadores clave (**KPIs**) antes de graficar.

La función más importante es **`groupby().agg()`**:
```
df.groupby('columna_agrupadora').agg(
    nuevo_nombre = ('columna_origen', 'funcion')
)
```
Funciones disponibles: `'sum'`, `'mean'`, `'count'`, `'min'`, `'max'`, `'std'`

In [ ]:
# ─────────────────────────────────────────────────────────────
# KPI 1 — Tasa de entregas exitosas
# Filtramos filas donde estado == 'Entregado' y contamos
# ─────────────────────────────────────────────────────────────
total_envios   = len(df)
entregados     = len(df[df['estado'] == 'Entregado'])   # filtro booleano
tasa_entrega   = round(entregados / total_envios * 100, 1)
costo_promedio = df['costo_envio_cop'].mean()
retraso_prom   = df['retraso_dias'].mean()

print(f'📦 Total envíos        : {total_envios}')
print(f'✅ Entregados          : {entregados}')
print(f'📊 Tasa de entrega     : {tasa_entrega}%')
print(f'💰 Costo promedio      : ${costo_promedio:,.0f} COP')
print(f'⏰ Retraso promedio    : {retraso_prom:.1f} días')

In [ ]:
# ─────────────────────────────────────────────────────────────
# KPI 2 — Resumen por transportista
# groupby agrupa todas las filas del mismo transportista
# agg() aplica funciones distintas a distintas columnas
# reset_index() convierte el índice agrupado en columna normal
# ─────────────────────────────────────────────────────────────
costo_transportista = (
    df.groupby('transportista')
      .agg(
          total_envios   = ('id_envio',        'count'),   # cuántos envíos
          costo_promedio = ('costo_envio_cop',  'mean'),   # costo promedio
          costo_total    = ('costo_envio_cop',  'sum'),    # costo acumulado
          retraso_prom   = ('retraso_dias',     'mean')    # retraso promedio
      )
      .round(0)
      .reset_index()
      .sort_values('costo_promedio', ascending=False)
)

print('🚛 Resumen por transportista:')
costo_transportista

In [ ]:
# ─────────────────────────────────────────────────────────────
# KPI 3 — Envíos por mes
# dt.to_period('M') convierte fecha a período mensual (2024-01, 2024-02...)
# Luego agrupamos por ese período
# ─────────────────────────────────────────────────────────────
envios_mes = (
    df.groupby(df['fecha_despacho'].dt.to_period('M'))
      .agg(total=('id_envio', 'count'))
      .reset_index()
)
# Convertimos el período a texto para graficarlo
envios_mes['fecha_despacho'] = envios_mes['fecha_despacho'].astype(str)

print('📅 Envíos por mes:')
envios_mes

---
## 📉 PASO 4 — Visualizaciones con Plotly Express

**Plotly Express** (`px`) crea gráficas interactivas con muy pocas líneas.

Estructura universal:
```python
fig = px.tipo_grafica(
    dataframe,          # datos a graficar
    x = 'columna_x',   # eje horizontal
    y = 'columna_y',   # eje vertical
    color = 'col',     # variable que define el color
    title = 'Título'
)
fig.update_layout(...)  # personalizaciones opcionales
fig.show()              # mostrar en Jupyter
# En Streamlit usarías: st.plotly_chart(fig)
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# GRÁFICA 1 — Pie chart: distribución de estados
# px.pie() → gráfico de torta
# names= columna con las etiquetas de cada sector
# values= columna con los valores numéricos de cada sector
# ─────────────────────────────────────────────────────────────
estado_counts = df['estado'].value_counts().reset_index()
# resultado: DataFrame con columnas 'estado' y 'count'

fig1 = px.pie(
    estado_counts,
    names  = 'estado',                              # etiquetas de sectores
    values = 'count',                               # tamaño de sectores
    title  = '📦 Distribución de Envíos por Estado',
    color_discrete_sequence = px.colors.qualitative.Set3  # paleta de colores
)
# update_layout personaliza el aspecto general de la figura
fig1.update_layout(title_font_size=16, showlegend=True)

fig1.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# GRÁFICA 2 — Barras: costo promedio por transportista
# px.bar() → gráfico de barras
# orientation='h' → barras horizontales (mejor para etiquetas largas)
# text_auto='.2s'  → muestra el valor encima/dentro de cada barra
# color_continuous_scale → escala de color para variable numérica
# ─────────────────────────────────────────────────────────────
fig2 = px.bar(
    costo_transportista.sort_values('costo_promedio', ascending=True),
    y          = 'transportista',
    x          = 'costo_promedio',
    orientation= 'h',
    title      = '💰 Costo Promedio por Transportista (COP)',
    labels     = {'costo_promedio': 'Costo Promedio (COP)', 'transportista': ''},
    color      = 'costo_promedio',
    color_continuous_scale = 'Blues',
    text_auto  = '.2s'
)
fig2.update_layout(height=350, showlegend=False)
fig2.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# GRÁFICA 3 — Scatter: peso vs costo
# px.scatter() → diagrama de dispersión
# size=  → tamaño del punto según variable numérica
# hover_data= → columnas extra que aparecen al pasar el mouse
# trendline='ols' → agrega línea de tendencia (regresión lineal)
# ─────────────────────────────────────────────────────────────
fig3 = px.scatter(
    df,
    x          = 'peso_kg',
    y          = 'costo_envio_cop',
    color      = 'tipo_carga',           # color diferente por tipo de carga
    size       = 'distancia_km',         # punto más grande = más distancia
    hover_data = ['transportista', 'ciudad_origen', 'ciudad_destino'],
    title      = '⚖️ Relación Peso vs Costo de Envío',
    labels     = {
        'peso_kg'        : 'Peso (Kg)',
        'costo_envio_cop': 'Costo (COP)',
        'tipo_carga'     : 'Tipo de Carga'
    },
    trendline  = 'ols'   # Ordinary Least Squares: línea de tendencia
)
fig3.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# GRÁFICA 4 — Línea: envíos por mes
# px.line() → ideal para series de tiempo (datos con evolución temporal)
# markers=True → muestra un punto en cada dato (más fácil de leer)
# update_traces → personaliza las líneas y marcadores
# ─────────────────────────────────────────────────────────────
fig4 = px.line(
    envios_mes,
    x       = 'fecha_despacho',
    y       = 'total',
    markers = True,                     # puntos visibles en cada mes
    title   = '📅 Envíos por Mes (2024)',
    labels  = {'fecha_despacho': 'Mes', 'total': 'Total de Envíos'}
)
# update_traces modifica las propiedades de las líneas/puntos
fig4.update_traces(line_color='#1f77b4', line_width=3, marker_size=10)
# update_xaxes rota las etiquetas del eje X para mejor legibilidad
fig4.update_xaxes(tickangle=-45)
fig4.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# GRÁFICA 5 — Heatmap: retrasos por origen y tipo de carga
# pivot_table() reorganiza los datos en forma de matriz:
#   filas    = ciudad_origen
#   columnas = tipo_carga
#   valores  = promedio de retraso_dias
# px.imshow() convierte esa matriz en un mapa de calor
# ─────────────────────────────────────────────────────────────
pivot = df.pivot_table(
    values  = 'retraso_dias',
    index   = 'ciudad_origen',
    columns = 'tipo_carga',
    aggfunc = 'mean'
).round(1)

fig5 = px.imshow(
    pivot,
    title                  = '🌡️ Retraso Promedio (días) por Ciudad y Tipo de Carga',
    color_continuous_scale = 'RdYlGn_r',  # Rojo=peor, Verde=mejor (invertido)
    text_auto              = True          # muestra el número en cada celda
)
fig5.update_layout(height=350)
fig5.show()

---
## 🌐 PASO 5 — Construcción de la App Streamlit

### ¿Qué es Streamlit?
Streamlit convierte un script Python en una aplicación web interactiva **sin necesidad de saber HTML, CSS ni JavaScript**. Cada vez que el usuario mueve un filtro, Streamlit re-ejecuta el script completo de arriba a abajo y actualiza la pantalla.

### Anatomía de una app Streamlit:

```
┌─────────────────────────────────────────────────────┐
│  st.set_page_config()  ← configuración inicial      │
│                                                     │
│  @st.cache_data        ← caché de datos             │
│  def cargar_datos():                                │
│      return pd.read_csv(...)                        │
│                                                     │
│  ┌──────────┐  ┌──────────────────────────────────┐ │
│  │ SIDEBAR  │  │         ÁREA PRINCIPAL           │ │
│  │          │  │  st.title()                      │ │
│  │ Filtros  │  │  ┌────┬────┬────┬────┐  KPIs    │ │
│  │          │  │  │col1│col2│col3│col4│           │ │
│  │ st.multi │  │  └────┴────┴────┴────┘           │ │
│  │ select() │  │  ┌──────────┐ ┌──────────┐       │ │
│  │          │  │  │ Gráfica 1│ │ Gráfica 2│       │ │
│  │ st.select│  │  └──────────┘ └──────────┘       │ │
│  │ box()    │  │  ┌─────────────────────────┐     │ │
│  └──────────┘  │  │      Heatmap completo   │     │ │
│                │  └─────────────────────────┘     │ │
│                └──────────────────────────────────┘ │
└─────────────────────────────────────────────────────┘
```

### Patrones de layout:
- **Patrón F:** El ojo escanea de izquierda a derecha en la parte superior (KPIs), luego baja y escanea menos. Por eso los KPIs van en la fila superior.
- **Patrón Z:** El ojo hace una Z — top izquierda → top derecha → diagonal → bottom izquierda → bottom derecha. Se usa en dashboards con gráficas alternando posición.


In [ ]:
# ─────────────────────────────────────────────────────────────
# BLOQUE A — Configuración inicial
# st.set_page_config() DEBE ser la primera llamada de Streamlit
#
# page_title  → texto de la pestaña del navegador
# page_icon   → emoji o URL de icono para la pestaña
# layout      → 'wide' usa todo el ancho | 'centered' centra el contenido
# initial_sidebar_state → 'expanded' abre el sidebar al cargar
# ─────────────────────────────────────────────────────────────

codigo_bloque_a = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import os

st.set_page_config(
    page_title            = 'TransCargo Dashboard',
    page_icon             = '🚚',
    layout                = 'wide',          # usa todo el ancho de la pantalla
    initial_sidebar_state = 'expanded'       # sidebar visible al cargar
)
'''
print('📄 Bloque A — Configuración inicial:')
print(codigo_bloque_a)

In [ ]:
# ─────────────────────────────────────────────────────────────
# BLOQUE B — Caché de datos
#
# @st.cache_data es un DECORADOR: modifica el comportamiento
# de la función que le sigue.
#
# Sin caché: cada vez que el usuario mueve un filtro,
#            Python lee el CSV desde disco → LENTO
# Con caché: la primera vez lee el CSV y lo guarda en memoria.
#            Las siguientes veces devuelve el resultado guardado → RÁPIDO
#
# os.path.dirname(__file__) → carpeta donde está el archivo .py
# os.path.join(...)         → construye la ruta completa al CSV
# Esto es necesario en Streamlit Cloud donde el directorio de
# trabajo puede ser diferente al del archivo .py
# ─────────────────────────────────────────────────────────────

codigo_bloque_b = '''
@st.cache_data
def cargar_datos():
    # Construir ruta relativa al archivo .py (funciona local y en Cloud)
    ruta = os.path.join(os.path.dirname(__file__), 'caso1_logistica_dataset.csv')
    df = pd.read_csv(ruta)
    df['fecha_despacho'] = pd.to_datetime(df['fecha_despacho'])
    return df

df = cargar_datos()   # ← llamar la función para obtener los datos
'''
print('📄 Bloque B — Caché de datos:')
print(codigo_bloque_b)

In [ ]:
# ─────────────────────────────────────────────────────────────
# BLOQUE C — Sidebar con filtros
#
# with st.sidebar: → todo lo que esté dentro aparece en el panel lateral
#
# st.multiselect() → lista desplegable con selección múltiple
#   options  = lista de opciones disponibles
#   default  = opciones seleccionadas al cargar (todas por defecto)
#
# st.selectbox() → lista desplegable con selección única
#
# sorted() → ordena las opciones alfabéticamente
# .unique() → obtiene valores únicos de la columna
# ─────────────────────────────────────────────────────────────

codigo_bloque_c = '''
with st.sidebar:
    st.header("🔧 Filtros")

    # Filtro 1: selección múltiple de transportistas
    transportistas_sel = st.multiselect(
        label   = "Transportista",
        options = sorted(df['transportista'].unique()),
        default = list(df['transportista'].unique())   # todos seleccionados
    )

    # Filtro 2: selección múltiple de tipo de carga
    tipos_carga_sel = st.multiselect(
        label   = "Tipo de Carga",
        options = sorted(df['tipo_carga'].unique()),
        default = list(df['tipo_carga'].unique())
    )

    # Filtro 3: selección única de estado
    estado_sel = st.selectbox(
        label   = "Estado del Envío",
        options = ['Todos'] + sorted(df['estado'].unique())
    )
'''
print('📄 Bloque C — Sidebar con filtros:')
print(codigo_bloque_c)

In [ ]:
# ─────────────────────────────────────────────────────────────
# BLOQUE D — Aplicar filtros al DataFrame
#
# df.copy() → crea una copia independiente del DataFrame original
#             para no modificar los datos cargados en caché
#
# .isin(lista) → filtra filas cuyo valor esté en la lista dada
#
# Lógica: si el usuario seleccionó algo en el filtro, lo aplica.
#         Si no seleccionó nada, no filtra (muestra todo).
# ─────────────────────────────────────────────────────────────

codigo_bloque_d = '''
df_f = df.copy()   # df_f = DataFrame filtrado

if transportistas_sel:
    df_f = df_f[df_f['transportista'].isin(transportistas_sel)]

if tipos_carga_sel:
    df_f = df_f[df_f['tipo_carga'].isin(tipos_carga_sel)]

if estado_sel != 'Todos':
    df_f = df_f[df_f['estado'] == estado_sel]
'''
print('📄 Bloque D — Aplicar filtros:')
print(codigo_bloque_d)

In [ ]:
# ─────────────────────────────────────────────────────────────
# BLOQUE E — Título y KPIs (patrón F)
#
# st.title()    → encabezado principal de la página (H1)
# st.markdown() → texto con formato Markdown (negritas, cursivas, etc.)
#
# st.columns(5) → divide el área en 5 columnas iguales
#   retorna 5 objetos columna a los que asignamos variables
#
# col.metric(label, value, delta) → tarjeta de KPI con:
#   label → texto descriptivo
#   value → valor principal (grande)
#   delta → variación (verde si positivo, rojo si negativo)
# ─────────────────────────────────────────────────────────────

codigo_bloque_e = '''
# ── Título principal ──────────────────────────────────────────
st.title("🚚 TransCargo — Dashboard de Operaciones")
st.markdown("**Panel de control de envíos · 2024**")
st.markdown("---")   # línea horizontal separadora

# ── KPIs en fila horizontal (patrón F) ───────────────────────
# El ojo escanea izquierda→derecha: los más importantes van primero
k1, k2, k3, k4, k5 = st.columns(5)

total      = len(df_f)
entregados = len(df_f[df_f['estado'] == 'Entregado'])
tasa       = round(entregados / total * 100, 1) if total > 0 else 0
costo_prom = df_f['costo_envio_cop'].mean() if total > 0 else 0
retraso    = df_f['retraso_dias'].mean() if total > 0 else 0
calific    = df_f['calificacion_cliente'].mean() if total > 0 else 0

k1.metric("📦 Total Envíos",     f"{total:,}")
k2.metric("✅ Tasa de Entrega",  f"{tasa}%")
k3.metric("💰 Costo Promedio",   f"${costo_prom:,.0f}")
k4.metric("⏰ Retraso Prom.",    f"{retraso:.1f} días")
k5.metric("⭐ Calificación",     f"{calific:.1f}/5")
'''
print('📄 Bloque E — Título y KPIs:')
print(codigo_bloque_e)

In [ ]:
# ─────────────────────────────────────────────────────────────
# BLOQUE F — Gráficas en layout Z (dos columnas)
#
# st.columns([1.5, 1]) → columna izquierda 1.5x más ancha que la derecha
#   Los números son proporciones relativas, no píxeles
#
# with col_izq: → todo lo que esté dentro va a la columna izquierda
#
# st.plotly_chart(fig, use_container_width=True)
#   → inserta la gráfica Plotly en la app
#   → use_container_width=True hace que ocupe el ancho completo de su columna
#
# Patrón Z: fila 1 → grande izq + pequeña der
#           fila 2 → pequeña izq + grande der  (alternado)
# ─────────────────────────────────────────────────────────────

codigo_bloque_f = '''
# ── Fila 1: línea de tiempo (grande) + pie (pequeño) ─────────
col_izq, col_der = st.columns([1.5, 1])

with col_izq:
    envios_mes = (
        df_f.groupby(df_f['fecha_despacho'].dt.to_period('M'))
            .agg(total=('id_envio', 'count'))
            .reset_index()
    )
    envios_mes['fecha_despacho'] = envios_mes['fecha_despacho'].astype(str)

    fig_linea = px.line(
        envios_mes, x='fecha_despacho', y='total',
        markers=True, title='📅 Envíos por Mes',
        color_discrete_sequence=['#2d6a9f']
    )
    fig_linea.update_traces(line_width=3, marker_size=8)
    st.plotly_chart(fig_linea, use_container_width=True)

with col_der:
    estado_counts = df_f['estado'].value_counts().reset_index()
    fig_pie = px.pie(
        estado_counts, names='estado', values='count',
        title='📦 Estado de Envíos',
        color_discrete_sequence=px.colors.qualitative.Set3
    )
    st.plotly_chart(fig_pie, use_container_width=True)

# ── Fila 2: barras (pequeño) + scatter (grande) ───────────────
col_izq2, col_der2 = st.columns([1, 1.5])

with col_izq2:
    cost_t = df_f.groupby('transportista')['costo_envio_cop'].mean().reset_index()
    cost_t.columns = ['transportista', 'costo_promedio']
    fig_bar = px.bar(
        cost_t.sort_values('costo_promedio'), y='transportista', x='costo_promedio',
        orientation='h', title='💰 Costo por Transportista',
        color='costo_promedio', color_continuous_scale='Blues', text_auto='.2s'
    )
    st.plotly_chart(fig_bar, use_container_width=True)

with col_der2:
    fig_sc = px.scatter(
        df_f, x='peso_kg', y='costo_envio_cop',
        color='tipo_carga', size='distancia_km',
        title='⚖️ Peso vs Costo', trendline='ols'
    )
    st.plotly_chart(fig_sc, use_container_width=True)
'''
print('📄 Bloque F — Gráficas en layout Z:')
print(codigo_bloque_f)

In [ ]:
# ─────────────────────────────────────────────────────────────
# BLOQUE G — Componentes adicionales
#
# st.expander("texto") → sección colapsable/expandible
#   Útil para tablas de datos o información secundaria
#   No ocupa espacio visual hasta que el usuario la abre
#
# st.dataframe(df) → muestra una tabla interactiva
#   use_container_width=True → ocupa todo el ancho disponible
#
# st.download_button() → botón para descargar archivo
#   label  → texto del botón
#   data   → contenido a descargar (df.to_csv() convierte a texto CSV)
#   file_name → nombre del archivo descargado
# ─────────────────────────────────────────────────────────────

codigo_bloque_g = '''
# ── Heatmap completo (ancho total) ───────────────────────────
pivot = df_f.pivot_table(
    values='retraso_dias', index='ciudad_origen',
    columns='tipo_carga', aggfunc='mean'
).round(1)

fig_heat = px.imshow(pivot, color_continuous_scale='RdYlGn_r',
                     text_auto=True, title='🌡️ Retraso Promedio por Ciudad y Tipo de Carga')
st.plotly_chart(fig_heat, use_container_width=True)

# ── Tabla de datos colapsable ─────────────────────────────────
with st.expander("📋 Ver datos filtrados"):
    st.dataframe(df_f.sort_values('fecha_despacho', ascending=False),
                 use_container_width=True)
    # Botón de descarga del dataset filtrado como CSV
    st.download_button(
        label     = "⬇️ Descargar CSV",
        data      = df_f.to_csv(index=False),
        file_name = "logistica_filtrado.csv"
    )

# ── Pie de página ─────────────────────────────────────────────
st.caption("🔧 Desarrollado con Streamlit + Plotly | Clase de Visualización de Datos")
'''
print('📄 Bloque G — Componentes adicionales:')
print(codigo_bloque_g)

---
## 🚀 PASO 6 — Ejecutar la app completa

Ahora unimos todos los bloques en el archivo `caso1_logistica_app.py` y lo ejecutamos.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Generar el archivo .py completo desde el notebook
# ─────────────────────────────────────────────────────────────
app_completa = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import os

# ① Configuración
st.set_page_config(page_title='TransCargo Dashboard', page_icon='🚚',
                   layout='wide', initial_sidebar_state='expanded')

# ② Carga de datos con caché
@st.cache_data
def cargar_datos():
    ruta = os.path.join(os.path.dirname(__file__), 'caso1_logistica_dataset.csv')
    df = pd.read_csv(ruta)
    df['fecha_despacho'] = pd.to_datetime(df['fecha_despacho'])
    return df

df = cargar_datos()

# ③ Sidebar con filtros
with st.sidebar:
    st.header("🔧 Filtros")
    transportistas_sel = st.multiselect("Transportista",
        sorted(df['transportista'].unique()), list(df['transportista'].unique()))
    tipos_carga_sel = st.multiselect("Tipo de Carga",
        sorted(df['tipo_carga'].unique()), list(df['tipo_carga'].unique()))
    estado_sel = st.selectbox("Estado", ['Todos'] + sorted(df['estado'].unique()))
    origen_sel = st.multiselect("Ciudad Origen",
        sorted(df['ciudad_origen'].unique()), list(df['ciudad_origen'].unique()))

# ④ Aplicar filtros
df_f = df.copy()
if transportistas_sel: df_f = df_f[df_f['transportista'].isin(transportistas_sel)]
if tipos_carga_sel:    df_f = df_f[df_f['tipo_carga'].isin(tipos_carga_sel)]
if estado_sel != 'Todos': df_f = df_f[df_f['estado'] == estado_sel]
if origen_sel:         df_f = df_f[df_f['ciudad_origen'].isin(origen_sel)]

# ⑤ Título
st.title("🚚 TransCargo — Dashboard de Operaciones Logísticas")
st.markdown("**Panel de control de envíos · 2024**")
st.markdown("---")

# ⑥ KPIs (patrón F)
k1, k2, k3, k4, k5 = st.columns(5)
total      = len(df_f)
entregados = len(df_f[df_f['estado'] == 'Entregado'])
tasa       = round(entregados / total * 100, 1) if total > 0 else 0
k1.metric("📦 Total Envíos",    f"{total:,}")
k2.metric("✅ Tasa Entrega",    f"{tasa}%")
k3.metric("💰 Costo Promedio",  f"${df_f['costo_envio_cop'].mean():,.0f}" if total > 0 else "$0")
k4.metric("⏰ Retraso Prom.",   f"{df_f['retraso_dias'].mean():.1f} días" if total > 0 else "0")
k5.metric("⭐ Calificación",    f"{df_f['calificacion_cliente'].mean():.1f}/5" if total > 0 else "0")
st.markdown("---")

# ⑦ Fila 1: línea + pie (patrón Z)
col1, col2 = st.columns([1.5, 1])
with col1:
    em = df_f.groupby(df_f['fecha_despacho'].dt.to_period('M')).agg(total=("id_envio","count")).reset_index()
    em['fecha_despacho'] = em['fecha_despacho'].astype(str)
    fig = px.line(em, x='fecha_despacho', y='total', markers=True,
                  title='📅 Envíos por Mes', color_discrete_sequence=['#2d6a9f'])
    fig.update_traces(line_width=3, marker_size=8)
    st.plotly_chart(fig, use_container_width=True)
with col2:
    ec = df_f['estado'].value_counts().reset_index()
    fig2 = px.pie(ec, names='estado', values='count', title='📦 Estado de Envíos',
                  color_discrete_sequence=px.colors.qualitative.Set3)
    st.plotly_chart(fig2, use_container_width=True)

# ⑧ Fila 2: barras + scatter (patrón Z invertido)
col3, col4 = st.columns([1, 1.5])
with col3:
    ct = df_f.groupby('transportista')['costo_envio_cop'].mean().reset_index()
    ct.columns = ['transportista','costo_promedio']
    fig3 = px.bar(ct.sort_values('costo_promedio'), y='transportista', x='costo_promedio',
                  orientation='h', title='💰 Costo por Transportista',
                  color='costo_promedio', color_continuous_scale='Blues', text_auto='.2s')
    fig3.update_layout(showlegend=False)
    st.plotly_chart(fig3, use_container_width=True)
with col4:
    fig4 = px.scatter(df_f, x='peso_kg', y='costo_envio_cop', color='tipo_carga',
                      size='distancia_km', hover_data=['transportista','ciudad_origen','ciudad_destino'],
                      title='⚖️ Peso vs Costo por Tipo de Carga', trendline='ols')
    st.plotly_chart(fig4, use_container_width=True)

# ⑨ Heatmap ancho completo
st.markdown("### 🌡️ Mapa de Calor — Retraso por Origen y Tipo de Carga")
pivot = df_f.pivot_table(values='retraso_dias', index='ciudad_origen',
                          columns='tipo_carga', aggfunc='mean').round(1)
fig5 = px.imshow(pivot, color_continuous_scale='RdYlGn_r', text_auto=True)
st.plotly_chart(fig5, use_container_width=True)

# ⑩ Tabla colapsable
with st.expander("📋 Ver datos filtrados"):
    st.dataframe(df_f.sort_values('fecha_despacho', ascending=False), use_container_width=True)
    st.download_button("⬇️ Descargar CSV", df_f.to_csv(index=False), "logistica_filtrado.csv")

st.caption("🔧 Streamlit + Plotly | Clase de Visualización de Datos")
'''

with open('caso1_logistica_app.py', 'w', encoding='utf-8') as f:
    f.write(app_completa)

print('✅ caso1_logistica_app.py generado exitosamente')
print('▶️  Ejecuta en Anaconda Prompt:')
print('   streamlit run caso1_logistica_app.py')

---
## ✅ RESUMEN — Componentes Streamlit aprendidos

| Función | Qué hace |
|---|---|
| `st.set_page_config()` | Configura título, ícono y layout de la página |
| `@st.cache_data` | Almacena datos en caché para no recargar en cada interacción |
| `st.sidebar` | Panel lateral para filtros y controles |
| `st.multiselect()` | Lista con selección múltiple |
| `st.selectbox()` | Lista con selección única |
| `st.title()` | Título principal de la página |
| `st.markdown()` | Texto con formato Markdown |
| `st.columns(N)` | Divide el área en N columnas |
| `st.metric()` | Tarjeta KPI con valor y delta |
| `st.plotly_chart()` | Inserta gráfica Plotly |
| `st.expander()` | Sección colapsable |
| `st.dataframe()` | Tabla interactiva |
| `st.download_button()` | Botón para descargar archivo |
| `st.caption()` | Texto pequeño de pie de página |

---
### ▶️ Para ejecutar la app:
```bash
streamlit run caso1_logistica_app.py
```